In [1]:
!pip install laspy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.3/84.3 kB 2.1 MB/s eta 0:00:00


In [3]:
import geopandas as gpd
import numpy as np
import laspy
import os
from sklearn.model_selection import train_test_split
import scipy.spatial
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv3D, MaxPooling3D, Flatten, Dense
from tensorflow.keras.optimizers import Adam

In [4]:
# Step 1.1: Load Field Inventory Data
field_inventory = gpd.read_file("field_survey.geojson")
print(field_inventory.head())

# Analyze species distribution
species_counts = field_inventory["species"].value_counts().reset_index()
species_counts.columns = ["species", "count"]
print(species_counts)


   plot  tree_no species    d1    d2    dbh   age  height  angle comment  \
0   1.0      1.0   Birch  47.2  46.2  46.70   NaN    26.5    0.0    None   
1   1.0      2.0   Aspen  27.9  29.1  28.50   NaN     NaN    0.0    None   
2   1.0      3.0     Fir  12.1  13.0  12.55   NaN     NaN    0.0    None   
3   1.0      4.0   Aspen  23.3  23.7  23.50  69.0    27.4    0.0    None   
4   1.0      5.0   Aspen  23.2  31.3  27.25   NaN     NaN    0.0    None   

                         geometry  
0   POINT (547075.84 6450425.243)  
1  POINT (547074.299 6450419.542)  
2  POINT (547077.454 6450419.994)  
3   POINT (547078.211 6450419.02)  
4  POINT (547074.669 6450415.573)  
  species  count
0  Spruce   1144
1   Birch    676
2     Fir    663
3   Aspen    598
4   Tilia    370
5   Alder    121
6  Willow     28
7     Elm      1
8    Pine      1


In [5]:
# List of individual .las files
las_files = [f for f in os.listdir() if f.endswith(".las")]

print(f"Found {len(las_files)} LAS files.")

# Function to preprocess a single LAS file
def preprocess_las(file_path, height_threshold=2.0):
    print(f"Processing {file_path}...")
    las = laspy.read(file_path)

    # Extract x, y, z coordinates
    points = np.vstack((las.x, las.y, las.z)).T

    # Normalize Z (height normalization)
    z_min = np.min(points[:, 2])
    points[:, 2] -= z_min

    # Filter points above height threshold
    filtered_points = points[points[:, 2] > height_threshold]

    return filtered_points

# Preprocess all LAS files
all_points = []
for las_file in las_files:
    filtered_points = preprocess_las(las_file)
    all_points.append(filtered_points)

# Combine all points into a single dataset
all_points = np.vstack(all_points)
print(f"Total points after filtering: {all_points.shape[0]}")

# Split dataset into training and testing sets
train_points, test_points = train_test_split(all_points, test_size=0.2, random_state=42)
print(f"Training set size: {train_points.shape[0]} points")
print(f"Testing set size: {test_points.shape[0]} points")

Found 10 LAS files.
Processing plot_10.las...
Processing plot_02.las...
Processing plot_08.las...
Processing plot_04.las...
Processing plot_09.las...
Processing plot_01.las...
Processing plot_05.las...
Processing plot_03.las...
Processing plot_06.las...
Processing plot_07.las...
Total points after filtering: 2698308
Training set size: 2158646 points
Testing set size: 539662 points


In [6]:
#STEP 2
def local_maxima_filter(cloud: np.ndarray, window_size: float | int, height_threshold: float | int) -> np.ndarray:
    """Detect local maxima in the point cloud with a fixed window size."""
    assert isinstance(cloud, np.ndarray), f"Cloud needs to be a numpy array, not {type(cloud)}"

    cloud = cloud[cloud[:, 2] > height_threshold]
    tree = scipy.spatial.KDTree(data=cloud)
    seen_mask = np.zeros(cloud.shape[0], dtype=bool)
    local_maxima = []

    for i, point in enumerate(cloud):
        if seen_mask[i]:
            continue
        neighbor_indices = tree.query_ball_point(point, window_size)
        highest_neighbor = neighbor_indices[np.argmax(cloud[neighbor_indices, 2])]
        seen_mask[neighbor_indices] = True
        seen_mask[highest_neighbor] = False
        # This may lead to not every point being marked as seed in the end, but it does not matter,
        # because by the time the seen value is overwritten the point is already processed
        if i == highest_neighbor:
            local_maxima.append(i)

    return cloud[local_maxima]


In [7]:
window_size = 3  # Increased window size for better detection
height_threshold = 0.5  # Lowered height threshold
filtered_maxima = local_maxima_filter(all_points, window_size, height_threshold)
print(f"Detected {len(filtered_maxima)} local maxima points.")



Detected 2986 local maxima points.


In [8]:
# Extract tree count from the field inventory
inventory_tree_count = len(field_inventory)
print(f"Tree count from field inventory: {inventory_tree_count}")

# Compare detected maxima points to field inventory
detected_maxima_count = len(filtered_maxima)
print(f"Detected local maxima points: {detected_maxima_count}")

Tree count from field inventory: 3602
Detected local maxima points: 2986


In [9]:
input_shape = (32, 32, 32, 1)  # Example voxel grid size

model = Sequential([
    Conv3D(32, (3, 3, 3), activation='relu', input_shape=input_shape),
    MaxPooling3D(pool_size=(2, 2, 2)),
    Conv3D(64, (3, 3, 3), activation='relu'),
    MaxPooling3D(pool_size=(2, 2, 2)),
    Flatten(),
    Dense(128, activation='relu'),
    Dense(1, activation='sigmoid')
])

model.compile(optimizer=Adam(learning_rate=0.001), loss='binary_crossentropy', metrics=['accuracy'])

/usr/local/lib/python3.10/dist-packages/keras/src/layers/convolutional/base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [11]:
field_inventory["x"] = field_inventory.geometry.x
field_inventory["y"] = field_inventory.geometry.y


In [12]:
ground_truth_locations = field_inventory[["x", "y"]].to_numpy()
detected_tree_locations = filtered_maxima[:, :2]

# Match detected trees to ground truth
ground_truth_tree = scipy.spatial.KDTree(data=ground_truth_locations)
matches = ground_truth_tree.query_ball_point(detected_tree_locations, r=1.0)  # Adjust radius as needed

# Calculate true positives, false positives, and false negatives
true_positive = np.array([len(match) > 0 for match in matches])
false_positive = ~true_positive
false_negative = len(ground_truth_locations) - sum(true_positive)

# Calculate performance metrics
precision = sum(true_positive) / (sum(true_positive) + sum(false_positive))
recall = sum(true_positive) / len(ground_truth_locations)
f1 = 2 * (precision * recall) / (precision + recall)

# Calculate average distance error
matched_distances = []
for idx, match in enumerate(matches):
    if len(match) > 0:
        detected_point = detected_tree_locations[idx]
        ground_truth_point = ground_truth_locations[match[0]]
        distance = np.linalg.norm(detected_point - ground_truth_point)
        matched_distances.append(distance)

average_distance_error = np.mean(matched_distances) if matched_distances else None

# Report results
print(f"Precision: {precision:.2f}")
print(f"Recall: {recall:.2f}")
print(f"F1-Score: {f1:.2f}")
if average_distance_error is not None:
    print(f"Average Distance Error: {average_distance_error:.2f} meters")
else:
    print("No matches found to calculate distance error.")


Precision: 0.31
Recall: 0.26
F1-Score: 0.28
Average Distance Error: 0.59 meters


In [24]:
def voxelize_point_cloud(points, grid_size=32):
    """Convert point cloud to a voxel grid."""
    # Normalize coordinates to [0, 1]
    min_coords = np.min(points, axis=0)
    max_coords = np.max(points, axis=0)
    epsilon = 1e-6  # Small value to prevent division by zero
    normalized_points = (points - min_coords) / (max_coords - min_coords + epsilon)

    # Remove points with NaN or invalid values
    valid_mask = np.all(np.isfinite(normalized_points), axis=1)
    normalized_points = normalized_points[valid_mask]

    # Scale to grid size and cast to integers
    voxel_coords = np.clip((normalized_points[:, :3] * grid_size).astype(int), 0, grid_size - 1)
    voxel_grid = np.zeros((grid_size, grid_size, grid_size), dtype=np.float32)

    # Fill voxel grid
    for x, y, z in voxel_coords:
        voxel_grid[x, y, z] = 1

    return voxel_grid

In [27]:
def prepare_voxelized_data(points, labels, grid_size=32):
    """Prepare balanced voxelized data for training/testing."""
    voxel_data = []
    voxel_labels = []

    for label, group in labels.groupby("plot"):
        # Points for the current plot
        plot_points = points[points[:, -1] == label]

        # Skip if no points are found for the plot
        if plot_points.size == 0:
            continue

        # Tree-present voxel grid
        voxel_grid = voxelize_point_cloud(plot_points[:, :3], grid_size=grid_size)
        voxel_data.append(voxel_grid)
        voxel_labels.append(1)

        # Tree-absent voxel grid (sample random noise)
        noise_points = np.random.uniform(
            np.min(points, axis=0), np.max(points, axis=0), (len(plot_points), 3)
        )
        noise_voxel_grid = voxelize_point_cloud(noise_points, grid_size=grid_size)
        voxel_data.append(noise_voxel_grid)
        voxel_labels.append(0)

    if len(voxel_data) == 0:
        raise ValueError("No valid data generated. Check input points and labels.")

    return np.array(voxel_data), np.array(voxel_labels)

# Call the function again after fixing
voxel_data, voxel_labels = prepare_voxelized_data(all_points, field_inventory)


In [44]:
voxel_data, voxel_labels = prepare_voxelized_data(all_points, field_inventory)
X_train, X_test, y_train, y_test = train_test_split(voxel_data, voxel_labels, test_size=0.2, random_state=42,shuffle=True)


In [45]:
model.fit(X_train, y_train, epochs=10, batch_size=16, validation_split=0.2)

# Evaluate the 3D CNN model
y_pred = (model.predict(X_test) > 0.2).astype(int)


Epoch 1/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step - accuracy: 1.0000 - loss: 0.0107 - val_accuracy: 1.0000 - val_loss: 0.0042
Epoch 2/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step - accuracy: 1.0000 - loss: 0.0090 - val_accuracy: 1.0000 - val_loss: 0.0034
Epoch 3/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step - accuracy: 1.0000 - loss: 0.0075 - val_accuracy: 1.0000 - val_loss: 0.0027
Epoch 4/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step - accuracy: 1.0000 - loss: 0.0064 - val_accuracy: 1.0000 - val_loss: 0.0022
Epoch 5/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step - accuracy: 1.0000 - loss: 0.0054 - val_accuracy: 1.0000 - val_loss: 0.0018
Epoch 6/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step - accuracy: 1.0000 - loss: 0.0046 - val_accuracy: 1.0000 - val_loss: 0.0014
Epoch 7/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step - accuracy: 1.0000 - loss: 0.0040 - val_accuracy: 1.0000 - val_loss: 0.0012
Epoch 8/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step - accuracy: 1.0000 - loss: 0.0035 - val_accuracy: 1.0000 - val_loss: 9.9567e-04
Epoch 9/10
1

In [46]:
from sklearn.metrics import precision_score, recall_score, f1_score

cnn_precision = precision_score(y_test, y_pred)
cnn_recall = recall_score(y_test, y_pred)
cnn_f1 = f1_score(y_test, y_pred)

# Compare with baseline metrics
print(f"3D CNN Metrics:")
print(f"Precision: {cnn_precision:.2f}, Recall: {cnn_recall:.2f}, F1-Score: {cnn_f1:.2f}")

print(f"Baseline Metrics:")
print(f"Precision: {precision:.2f}, Recall: {recall:.2f}, F1-Score: {f1:.2f}")

# Analyze and visualize results

3D CNN Metrics:
Precision: 1.00, Recall: 1.00, F1-Score: 1.00
Baseline Metrics:
Precision: 0.31, Recall: 0.26, F1-Score: 0.28
